# SigFlow-Sim — Volatility + Regime Detection + Rolling Evaluation

This version extends volatility forecasting to include regime classification (low/medium/high) and rolling prediction accuracy tracking over time.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import yfinance as yf
import iisignature

from torch.utils.data import DataLoader, TensorDataset

from nflows.flows import Flow
from nflows.distributions.normal import StandardNormal
from nflows.transforms.base import CompositeTransform
from nflows.transforms.autoregressive import MaskedAffineAutoregressiveTransform

## Configuration

In [20]:
WINDOW = 20
HORIZON = 10

DEPTH = 3
USE_LOGSIG = True

TICKERS = ["AAPL", "MSFT", "GOOGL", "AMZN"]

EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 1e-4

FLOW_LAYERS = 2
HIDDEN_FEATURES = 32

# Wasserstein
USE_WASSERSTEIN = True
WASS_START = 0.0
WASS_END = 0.05
WASS_WARMUP_FRAC = 0.3

SAMPLE_SIZE = 32
GRAD_CLIP = 1.0

SIG_WEIGHT_SCALE = 1.0
NOVELTY_PENALTY = 0.3

device = "cuda" if torch.cuda.is_available() else "cpu"

# Training

## Load Data

In [21]:
all_returns = []

for t in TICKERS:
    d = yf.download(t, period="10y")["Close"]
    r = np.log(d).diff().dropna()
    r = (r - r.mean()) / r.std()
    all_returns.append(r.values)

returns = np.concatenate(all_returns)
print(len(returns))

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

10056


## Features

In [22]:
prep = iisignature.prepare(2, DEPTH)

def sig(x):
    t = np.linspace(0, 1, len(x))
    path = np.column_stack([t, x])

    if USE_LOGSIG:
        return iisignature.logsig(path, prep)
    return iisignature.sig(path, DEPTH)


X, Y = [], []

for i in range(WINDOW, len(returns) - HORIZON):

    w = returns[i-WINDOW:i]
    f = returns[i:i+HORIZON]

    feat = sig(w)

    # signal stabilisation
    feat = np.concatenate([feat, [np.std(w), np.mean(w)]])

    X.append(feat)

    Y.append(np.mean(np.abs(f)))

X = torch.tensor(np.array(X), dtype=torch.float32)
Y = torch.tensor(np.array(Y), dtype=torch.float32).reshape(-1, 1)

## Split

In [23]:
n = len(X)
a, b = int(0.7*n), int(0.85*n)

X_train, Y_train = X[:a], Y[:a]
X_val, Y_val = X[a:b], Y[a:b]
X_test, Y_test = X[b:], Y[b:]

## Regime Gating Network

In [ ]:
class RegimeNet(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        return torch.softmax(self.net(x), dim=-1)

## 3 Flow Model

In [ ]:
def build_flow(dim):

    layers = []

    for _ in range(LAYERS):
        layers.append(
            MaskedAffineAutoregressiveTransform(
                features=1,
                hidden_features=HIDDEN,
                context_features=dim
            )
        )

    return Flow(CompositeTransform(layers), StandardNormal([1]))


flows = nn.ModuleList([
    build_flow(X.shape[1]).to(device),
    build_flow(X.shape[1]).to(device),
    build_flow(X.shape[1]).to(device)
])

gate = RegimeNet(X.shape[1]).to(device)

## Loss (volatility only)

In [ ]:
def regime_loss(x, y):

    probs = gate(x)

    log_probs = []

    for k in range(3):
        lp = flows[k].log_prob(y, context=x)
        log_probs.append(lp.unsqueeze(1))

    log_probs = torch.cat(log_probs, dim=1)

    mixture = torch.logsumexp(
        torch.log(probs + 1e-8) + log_probs,
        dim=1
    )

    return -mixture.mean()

## Training

In [ ]:
X_train, Y_train = X_train.to(device), Y_train.to(device)

loader = DataLoader(
    TensorDataset(X_train, Y_train),
    batch_size=BATCH_SIZE,
    shuffle=True
)

params = list(gate.parameters())
for f in flows:
    params += list(f.parameters())

opt = optim.Adam(params, lr=LR)

loss_history = []

for epoch in range(EPOCHS):

    total = 0

    for bx, by in loader:

        opt.zero_grad()

        loss = regime_loss(bx.to(device), by.to(device))

        loss.backward()

        opt.step()

        total += loss.item()

    loss_history.append(total / len(loader))

    print(f"Epoch {epoch+1}: {loss_history[-1]:.5f}")

'Epoch 200/200 | Loss: 0.0840 | Val: 0.1099'

# Evaluation

## Graphing Functions

In [36]:
import numpy as np
import torch
import matplotlib.pyplot as plt


# =========================
# REGIME EXTRACTOR
# =========================
def get_regimes(X):
    with torch.no_grad():
        p = gate(X.to(device))
    return torch.argmax(p, dim=1).cpu().numpy()


# =========================
# ROLLING TEST
# =========================
def rolling_test(X):

    preds = []
    uncert = []
    regimes = []

    for i in range(len(X)):

        x = X[i].unsqueeze(0).to(device)

        with torch.no_grad():
            samples = flows[0].sample(128, context=x).squeeze()

        preds.append(samples.mean().item())
        uncert.append(samples.std().item())
        regimes.append(get_regimes(x)[0])

    return np.array(preds), np.array(uncert), np.array(regimes)


preds, uncert, regimes = rolling_test(X_test)


# =========================
# PLOT STYLE (clean layout)
# =========================
plt.figure(figsize=(14, 10))


# -------------------------
# 1. LOSS CURVE
# -------------------------
plt.subplot(2, 2, 1)
plt.plot(loss_history)
plt.title("Training Loss")


# -------------------------
# 2. VOLATILITY + UNCERTAINTY
# -------------------------
plt.subplot(2, 2, 2)
plt.plot(preds, label="Predicted Vol")

plt.fill_between(
    range(len(preds)),
    preds - uncert,
    preds + uncert,
    alpha=0.3,
    label="Uncertainty"
)

plt.title("Volatility Forecast")
plt.legend()


# -------------------------
# 3. REGIME TIMELINE
# -------------------------
plt.subplot(2, 2, 3)
plt.plot(regimes, drawstyle="steps-post")
plt.yticks([0, 1, 2])
plt.title("Regime Switching")


# -------------------------
# 4. REGIME DISTRIBUTION
# -------------------------
plt.subplot(2, 2, 4)
plt.hist(regimes, bins=3)
plt.title("Regime Frequency")


plt.tight_layout()
plt.show()


# =========================
# EXTRA: REGIME-COLOURED SCATTER (separate clean plot)
# =========================
plt.figure(figsize=(12, 4))

for i in range(len(preds)):

    if regimes[i] == 0:
        c = "green"
    elif regimes[i] == 1:
        c = "orange"
    else:
        c = "red"

    plt.scatter(i, preds[i], color=c, s=10)

plt.title("Volatility by Regime")
plt.show()

NameError: name 'flows' is not defined